# Picking the hero commodity
Rank HS-2 chapters by re-export size, hub-intensity (rex ÷ imp), and
re-export share (rex ÷ total exports) to find a genuine transit commodity.

Resolve the project root (works whether the notebook runs from `notebooks/` or the project root), add it to `sys.path`, and import `HS2_NAMES` from `src.hs_reference` for labeling chapters later. Then load the three trade flows — imports, re-exports, and national (domestic) exports — from the processed parquet files under `data/processed`. Rows with zero or negative `value_usd` are dropped, and the row counts of each flow are printed as a sanity check.

In [18]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.hs_reference import HS2_NAMES

PROC = ROOT / "data" / "processed"

def load(flow):
    df = pd.read_parquet(PROC / f"{flow}.parquet")
    return df[df["value_usd"] > 0]

imp, rex, nat = load("import"), load("reexport"), load("national_export")
print("rows:", len(imp), len(rex), len(nat))

rows: 1758287 265687 113167


Aggregate each flow to the HS-2 chapter level and join them into one table `t`, then derive the two ranking metrics described above:

- **`hub_intensity`** = `rex_usd / imp_usd` — of everything imported in a chapter, what fraction comes back out as a re-export. A value near 1 (or above, since re-exports can lag/lead imports across years) means Bahrain is mainly a pass-through point for that commodity rather than a consumer of it.
- **`reexport_share`** = `rex_usd / (rex_usd + nat_usd)` — of everything Bahrain exports in that chapter (re-exports plus genuine domestic exports), what fraction is re-exported goods versus locally produced/processed goods. A value near 1 means the chapter's exports are almost entirely transit trade, not domestic production.

`top_commodity` grabs the most common commodity description among re-exports in each chapter. `chapter` then maps the HS-2 code to a human-readable chapter name via `HS2_NAMES`, falling back to `top_commodity` for any code missing from that lookup, so the final table is easier to read than raw HS-2 codes. Chapters with no imports get `NA` for `hub_intensity` instead of a divide-by-zero error, and the table is sorted by raw re-export value (`rex_usd`) so the biggest transit flows surface first.

In [20]:
def by_hs2(df, name):
    return df.groupby("hs2")["value_usd"].sum().rename(name)

t = pd.concat([by_hs2(imp,"imp_usd"), by_hs2(rex,"rex_usd"), by_hs2(nat,"nat_usd")], axis=1).fillna(0)

t["hub_intensity"]  = t["rex_usd"] / t["imp_usd"].replace(0, pd.NA)
t["reexport_share"] = t["rex_usd"] / (t["rex_usd"] + t["nat_usd"]).replace(0, pd.NA)

t["top_commodity"] = rex.groupby("hs2")["commodity"].agg(
    lambda s: s.mode().iat[0] if not s.mode().empty else "")

t["chapter"] = t.index.to_series().map(HS2_NAMES).fillna(t["top_commodity"])   # ← fixed line

t = t.sort_values("rex_usd", ascending=False)

Format dollar figures with thousands separators and two decimals, then display the top 20 HS-2 chapters by `rex_usd` alongside the import/national totals and both metrics, labeled by chapter name — this is the actual shortlist used to pick a "hero" transit commodity.

In [22]:
pd.options.display.float_format = lambda x: f"{x:,.2f}"
t.head(20)[["chapter","imp_usd","rex_usd","nat_usd","hub_intensity","reexport_share"]]

,chapter,imp_usd,rex_usd,nat_usd,hub_intensity,reexport_share
hs2,,,,,,
84,Machinery & mechanical appliances,"10,244,633,892.07","3,528,421,959.00","589,211,620.15",0.34,0.86
87,Vehicles & parts,"6,714,863,298.97","2,804,291,208.62","18,299,062.19",0.42,0.99
85,Electrical machinery & electronics,"6,362,415,845.40","1,196,063,288.32","83,974,437.14",0.19,0.93
71,"Pearls, gems & precious metals","4,366,381,176.47","1,104,141,937.25","1,362,022,786.99",0.25,0.45
91,Clocks & watches,"822,375,202.76","651,340,259.97","5,134,240.27",0.79,0.99
88,Aircraft & spacecraft,"150,619,991.95","392,146,935.90","185,808,639.02",2.60,0.68
83,Misc base metal articles,"336,903,505.44","269,144,324.80","7,352,534.52",0.80,0.97
98,Special/personal effects,"262,356,325.32","242,111,677.52","34,868,556.66",0.92,0.87
73,Iron/steel articles,"2,104,904,754.37","231,541,377.62","2,719,294,377.42",0.11,0.08


## Building corridors for the hero commodity
HS-87 (vehicles & parts) is picked as the hero chapter — it's large, almost entirely re-exported (`reexport_share` 0.99), and has solid hub-intensity. Filter the import and re-export tables down to just that chapter, and check how many distinct origin and destination countries are involved.

In [28]:
HERO = "87"
imp87 = imp[imp["hs2"] == HERO]
rex87 = rex[rex["hs2"] == HERO]
print("imp87 rows:", len(imp87), "| rex87 rows:", len(rex87))
print("origins:", imp87["country_iso2"].nunique(), "| destinations:", rex87["country_iso2"].nunique())

imp87 rows: 47491 | rex87 rows: 16754
origins: 151 | destinations: 115


The trade data has no shipment-level origin→destination link, so `build_corridors` estimates one via proportional allocation, done separately per month (`period`) since trade mix shifts over time:

1. Sum imports by origin country and re-exports by destination country for that month.
2. Turn each origin's import total into a share of that month's total imports (`shares`).
3. Take the outer product of `shares` (origins) and `rex_month` (destination totals) — this assumes each origin contributes to each destination in proportion to its share of imports, i.e. the destination mix of re-exports is applied uniformly across all origins.
4. Reshape the resulting origin×destination matrix into long form (`origin`, `dest`, `est_value`) and tag it with the period, then concatenate all months together.

This is a modeling assumption, not observed fact — it distributes re-exports across origins proportionally rather than tracking real shipments, so `est_value` should be read as an estimate of corridor volume, not ground truth.

In [38]:
import numpy as np

def build_corridors(imp_df, rex_df, value_col="value_usd"):
    """Estimate origin -> Bahrain -> destination flows via proportional allocation."""
    imp_po = imp_df.groupby(["period", "country_iso2"])[value_col].sum()
    rex_po = rex_df.groupby(["period", "country_iso2"])[value_col].sum()
    rex_periods = set(rex_po.index.get_level_values("period"))

    frames = []
    for period in imp_po.index.get_level_values("period").unique():
        if period not in rex_periods:
            continue
        imp_month = imp_po.loc[period]
        rex_month = rex_po.loc[period]
        total = imp_month.sum()
        if total <= 0:
            continue
        shares = imp_month / total

        mat = np.outer(shares.values, rex_month.values)
        block = pd.DataFrame(mat, index=shares.index, columns=rex_month.index)
        block.index.name = "origin"        # ← name the row axis
        block.columns.name = "dest"        # ← name the column axis
        block = block.stack().rename("est_value").reset_index()
        block["period"] = period
        frames.append(block)

    return pd.concat(frames, ignore_index=True)[["period", "origin", "dest", "est_value"]]


corridors = build_corridors(imp87, rex87)
print("corridor rows:", len(corridors))

corridors.sort_values("est_value", ascending=False).head(10)

corridor rows: 132411


,period,origin,dest,est_value
112403,2025-08-01,JP,AE,"17,695,879.73"
116007,2025-10-01,JP,AE,"11,604,172.41"
93226,2024-11-01,JP,AE,"9,980,812.12"
102956,2025-04-01,JP,AE,"9,297,875.28"
110110,2025-07-01,JP,AE,"9,086,223.74"
119774,2025-12-01,JP,AE,"8,999,361.21"
85291,2024-07-01,JP,SA,"8,947,328.21"
15204,2021-08-01,JP,AE,"8,700,413.41"
63982,2023-08-01,JP,AE,"8,642,965.15"
67665,2023-10-01,JP,AE,"8,454,595.34"


Build an ISO2 → country name lookup directly from the trade data (imports and re-exports both carry `country_name`), then sum the estimated corridor values across all months per `(origin, dest)` pair and take the 15 largest. Map both endpoints to readable names for the final display — Japan → UAE and Japan → Saudi Arabia dominate the estimated HS-87 transit corridors.

In [36]:
# iso2 -> readable name (from the data itself)
iso_name = (pd.concat([imp[["country_iso2","country_name"]], rex[["country_iso2","country_name"]]])
              .dropna().drop_duplicates("country_iso2").set_index("country_iso2")["country_name"])

top = (corridors.groupby(["origin","dest"])["est_value"].sum()
                .sort_values(ascending=False).head(15).reset_index())
top["origin_name"] = top["origin"].map(iso_name)
top["dest_name"]   = top["dest"].map(iso_name)
top[["origin_name","dest_name","est_value"]]

,origin_name,dest_name,est_value
0,Japan,United Arab Emirates,"350,000,029.45"
1,Japan,Saudi Arabia,"249,012,406.94"
2,China,United Arab Emirates,"137,249,058.44"
3,China,Saudi Arabia,"98,206,541.26"
4,Germany,United Arab Emirates,"87,485,709.45"
5,United States Of America,United Arab Emirates,"82,836,085.73"
6,Thailand,United Arab Emirates,"70,616,526.85"
7,Germany,Saudi Arabia,"61,777,406.64"
8,United States Of America,Saudi Arabia,"59,737,167.90"
9,Thailand,Saudi Arabia,"48,196,152.90"
